# Partner Finder — silnik AI (Faza 1)

Ocenia firmy pod kątem partnerstwa dla **Last Agency** (SEO/SEM/GEO, Poznań).

Napisane na OpenAI Agents SDK. Wszystko w jednym notebooku — na naukę.

## Importy

Wczytujemy `.env` (klucz `OPENAI_API_KEY`). `override=True` => `.env` wygrywa z ewentualną zmienną już ustawioną w systemie.

In [205]:
from dotenv import load_dotenv 
import os 
# importujemy sdk agent openai
from agents import Agent, Runner, trace, function_tool, SQLiteSession
# importujemy Basemodel do zmianany odpowiedzi evaluaotor ze string na obiekt i do utworzenia klasy evaluate
from pydantic import BaseModel

# importujemy csv do zapisu wyyników tools

import csv

# z bs4 importujemy Beautifoul soup do uporzodkowania pliku html i zamiany go z 1 wielkiego stringa na uporzadkowona strukutre czytelna dla pytona 
from bs4 import BeautifulSoup

# importujemy requests do wysyłania zapytań do serwera http
import requests 

## Narzędzia (tools)

UWAGA: w Agents SDK NIE piszemy ręcznie JSON-a opisującego narzędzie. Dekorator `@function_tool` sam generuje schemat z:
- nazwy funkcji,
- podpowiedzi typów (`url: str`),
- docstringa (opis + opis argumentów).

Dlatego **docstring jest teraz WAŻNY** — model go czyta, żeby wiedzieć, co robi tool.

In [206]:
# tutaj piszesz narzędzie scrape_website
# dajemy znacznik function_tool
@function_tool

# definiujemy funkcje z podpowiedziami typów jak string url object jako parametr przekazujemy url


def scrape_website(url : str) -> str:
# dajemy opis do description 
    """narzędzie skrapuje daną stronę i pobiernia z niej tekst , url pozwala scrapować strone"""

    # pobieramy strone i przekazujemy ją do zmienenj dzieki requests, pobranej bibliotece a następnie metodzie get, jako argument przegkazujemy url
    # dodajemy też timout, żeby strona nie ładowała sie w nieskonczonosc

    result =  requests.get(url,timeout=20)

    #print(result.text) # result pobiera htlm, teraz musimy wyciagjac text bez text do result przekazujemy sam status 200 - ze pobrano z serwera <Response [200]> w .text dostajmy cały html <!DOCTYPE html....

    # teraz chcemy wyciągnąć response text dzięki BeauftiulSoup, przekazujemy go do zmiennej responseText
    # dzieki bibliotece BeauftilSoup, 1 wielki string html pobrany dzieki request.text zamieniamy na czytelna dla pythona strukture stringów, pomoze nam to zebrac text z html odpowiednio pod kątem ulozenia danych
    #jako argument przekazujemy pborany html który mamy w result dzieki request.text, i wyciagamy zniego z result strngi metodą .text a Beautifoul soup nam  z 1 stringa robi wiele ustruktoryzwonaych na html 
    # 2 argument html.parser informuje pythona jak traktowac text jako html a metoda parser. to wbudowany moduł, funkcja pythona który pozwala zinterpetować jaki dokuemtnn czyta python
    # czyli metodą parser wskazujemy beuatiul soup zeby text z result czytac jako html
    responseText = BeautifulSoup(result.text, 'html.parser')

    # print(responseText) # pokazuje uporzadkowany html   z resuklt.text pobranymetdoą request.get dzieki beuafil sopu i html parser
    
    # teraz z uporzdkowane brzez beaufioul sop,html;a wyciagnietego z result.text dzieki request. get które polaczyło sie z sewerem http i pobrało html, a finalnie dzieki parser html soup woe jak czytac, metodą
    # getText wyciągy same stringi i przekazujemy do zmiennej

    finalText = responseText.get_text()


    # zwracamy wynik do nazwy funkcji, zeby to co ją wywoła moglo otrzymac text ze strony do scoringu

    return finalText[:5000]

In [207]:
# rovimy 2 narziedzie save_partner do csv

@function_tool


# piszemy funkcje z argumentami i dajemy ifo dla toola o typie daych

def save_partner(company : str, url : str, score : int , reason : str) -> str:

#piszemy opis 
    """narzędzie ma zapisać w pliku csv nazwe firmy, url, score i reason ,company ma zapisac nazwe firmy , url zapisuje url firmy, score podaje wynik scoringu, reason podaje powód dla którego mamy dany wynik i informacje o scoringu, dlacze
    go model podjał taka decyzje """

    # metodą with open otwieramy plik csv z metodą append do nadpisywania zapisujemy gi jako obiekt f

    with open('partners.csv' , 'a' , newline="" ,encoding="utf-8") as f:
        
        # teraz przekazujemy do zmiennej plik f dzieki bibliotece csv i metodzie writer, która pozwala nadpisywac plik, tówniez dzieki przekazaniu do obiektu f
        # pliku csv i metody append ' a narazie nie mamy nic zapisanego tylko samo narzedzie
        text = csv.writer(f)


        # teraz na text ktory jest narziedziem do zapisywani w csf, dzieki metodzie writer i przekazanoi do niej obiektu f który zawier nasz csv, metode append i inne
        #  uzyjemy metody writerow, która pozwoli nam zapisać wartosci przypisane do kazdego argumentu czyli nazwe, url, score i reason 

        finalText = text.writerow([company,url,score,reason])

        return 'zapisano'

In [208]:
# instrukcja dla agent i agent_rerun


SCORING_INSTRUCTIONS =  """Jesteś analitykiem partnerskim Last Agency - agencji SEO/SEM/GEO/AI Search z Poznania.
Wspierasz Growth & Partnerships Lead w budowie sieci partnerów: referral (polecają nam klientów) i white-label (zlecają nam usługi pod swoją marką).

ZADANIE: dostajesz URL firmy. Użyj scrape_website, oceń potencjał partnerski (1-10), i jeśli score >= 7, zapisz przez save_partner.

FLOW:
1. scrape_website na podanym URL
2. Rozpoznaj kategorię firmy i oceń wg kryteriów
3. Jeśli score >= 7 -> save_partner(company, url, score, reason)
4. Zwróć pełną ocenę

DOBRE KATEGORIE PARTNERÓW (firmy z tych obszarów = kandydaci na referral/white-label):
- Web dev / software house / twórcy sklepów (zwł. e-commerce)
- Agencje e-commerce (wdrożenia PrestaShop/Magento/Shopify/headless)
- Agencje brandingowe, kreatywne, social media
- Marketing automation (resellerzy/twórcy narzędzi)
- Digital advisory & delivery (doradztwo e-commerce, analityka, PM)
- Integratory CRM/ERP/PIM, CMS/SaaS e-commerce
- Kancelarie prawne e-commerce/RODO/AI
- Agencje AI/automatyzacje/chatboty, narzędzia GEO/AI-visibility
- Agencje growth/consulting, HR/employer branding, CRO
- Persony: doradcy biznesowi, dyrektorzy sprzedaży/marketingu, prawnicy e-commerce, osoby z siecią kontaktów

MOCNE SYGNAŁY (podbijają score do 8-10 - NIE asekuruj się szóstką jeśli je widzisz):
- E-commerce na premium platformach (PrestaShop, Magento, Shopify Plus) -> klienci z budżetem
- Oficjalny partner platformy (PrestaShop 3 gwiazdki, Shopify Partner) -> wiarygodność, top tier
- Dział marketingu / rozbudowana oferta -> gotowość na referral (wspólne migracje, prowizja = dodatkowe MRR)
- Naturalna synergia e-commerce <-> SEO: ich klienci (sklepy) potrzebują SEO/SEM
- Lider niszy / dużo case studies / wielu klientów -> skala i wolumen

PRZYKŁAD-KOTWICA:
Firma jak Tebim (sklepy PrestaShop, partner 3 gwiazdki, e-commerce, dział marketingu, top 3 w niszy) = score 9, model referral. Widzisz podobny profil -> oceniaj 8-10, nie 6.

ZŁY PARTNER (score 1-4):
- Agencja SEO/SEM/GEO/pozycjonowanie - bezpośredni konkurent
- Full-service 360 z własnym działem SEO
- Katalog/portal/agregator (Clutch, Sortlist)
- Freelancer bez klientów, firma spoza Polski
- Globalny gigant poza naszą skalą (np. worldline)

ŚREDNI (5-6): ma klientów, ale trochę SEO w ofercie (ryzyko konfliktu) LUB za mało danych/budżetu.
Uwaga: firma z SEO in-house NIE zawsze jest zła - może być sub-partnerem. Zaznacz to.

KRYTERIA KWALIFIKACJI:
- Brak konkurencyjnego SEO/SEM/GEO in-house (chyba że jako sub-partner)
- Portfolio klientów pasujące do profilu Last Agency
- Potencjał wolumenu (liczba klientów, częstotliwość projektów)
- Wiarygodność (opinie, lata działania, referencje)

W OCENIE ZAWRZYJ:
- score (1-10) + czy kontaktować (TAK/NIE)
- kategoria firmy (z listy powyżej)
- rekomendowany model: referral / white-label / oba
- czy mają SEO/SEM/GEO w ofercie (jeśli tak: konkurent czy sub-partner)
- szacowany profil klientów (wielkość, budżety)
- dane kontaktowe JEŚLI są na stronie (telefon, mail) - NIE zmyślaj, podaj tylko to, co faktycznie znalazłeś
- 3 zdania uzasadnienia oparte na treści strony

TON: konkretnie, bez lania wody, bez zbędnych disclaimerów."""

In [209]:
EVALUATOR_PROMPT = """Oceniasz JAKOŚĆ analizy partnerskiej wykonanej przez innego agenta AI dla Last Agency - agencji SEO/SEM/GEO z Poznania. NIE oceniasz firmy od nowa - sprawdzasz, czy ocena agenta jest solidna i zgodna z kryteriami.

CO SPRAWDZASZ:
1. Czy score jest poparty KONKRETNYMI faktami ze strony (nazwa usług, platformy, klienci), a nie ogólnikami.
2. Czy agent poprawnie rozpoznał, czy firma ma SEO/SEM/GEO w ofercie:
   - jeśli ma i dostała wysoki score jako "partner" (nie sub-partner) -> BŁĄD, to konkurent.
3. Czy agent NIE ZANIŻYŁ score przy widocznych MOCNYCH SYGNAŁACH. To częsty błąd - asekurowanie się szóstką.
   MOCNE SYGNAŁY, które powinny dać 8-10:
   - e-commerce na premium platformach (PrestaShop, Magento, Shopify Plus)
   - oficjalny partner platformy (np. PrestaShop 3 gwiazdki, Shopify Partner)
   - dział marketingu / rozbudowana oferta wdrożeniowa
   - naturalna synergia e-commerce <-> SEO, wielu klientów, lider niszy
   (Wzorzec: firma jak Tebim = 9. Jeśli agent widział taki profil i dał 6 -> BŁĄD, każ podnieść.)
4. Czy rekomendowany model (referral / white-label / oba) ma sens dla tej firmy.
5. Czy agent NIE ZMYŚLIŁ danych kontaktowych - kontakt (telefon/mail) jest OK tylko jeśli faktycznie był na stronie.
6. Czy uzasadnienie odnosi się do treści strony, a nie do ogólnych założeń.

ODRZUĆ (is_acceptable: false), jeśli:
- score 7+ dla konkurenta SEO/SEM/GEO (bez zaznaczenia "sub-partner")
- score zaniżony mimo wyraźnych mocnych sygnałów (np. premium e-commerce z partnerstwem platformy dostaje 6)
- uzasadnienie bez konkretów ze strony
- score 7+ dla firmy spoza Polski lub agregatora/katalogu
- zmyślone dane kontaktowe
- brak wskazania modelu współpracy przy wysokim score

ZATWIERDŹ (is_acceptable: true), jeśli:
- score wynika z konkretnych faktów ze strony
- poprawnie rozpoznano konkurenta vs partnera
- mocne sygnały właściwie zważone (nie zaniżone)
- uzasadnienie spójne ze score

W polu feedback napisz KONKRETNIE co jest nie tak i jak poprawić (np. "firma jest partnerem PrestaShop 3 gwiazdki - to mocny sygnał, score powinien być 8-9, nie 6"). Jeśli wszystko OK, feedback krótki: "ocena solidna".

TON: konkretnie, bez lania wody."""

In [210]:


# tworzymy agenta ten agent sam obsłuzy to co pisalismy wczesniej, czyli funkcje handle_tool petle while done itp
# w agencie podajmy name, czyli jego nazwe , instruction  czyli prompt,  tools, gdzie podajemy nazwy funkcji tools, i model
agent = Agent(

name = 'Partner Finder',
instructions= SCORING_INSTRUCTIONS,
tools = [scrape_website, save_partner],
model = 'gpt-5.4-mini'



)

In [211]:

# tworzymy klase do ewaluacji naszego agenta, klasa bierze odpowiedz modelu ewaluator i zamienia ja na obiekt z wartosciami string oraz bool,
# opart eo moduł z pydantica BaseModel

class Evaluator(BaseModel):
    is_acceptable : bool
    feedback : str

In [212]:

# piszemy teraz agenta Evalautor, który weryfikuje odpowiedz modeli agent i przeakzuje ją do klasy evalautor, klase przekazemy w outpout_type tam zwrocimu Evalautor output type wskazuje czym ma byc odpowiedz z modelu
# my chcemy obiekt zeby wykorzsytac Evalauoyta i jego 2 wlasnosci is_acceptable i feedback

agent_evaluator = Agent(
    name = 'evaluate',
    instructions= EVALUATOR_PROMPT,
output_type= Evaluator,
model = 'gpt-5.4-mini'


)


In [213]:

# piszemy agnet_rerun który odpali sie jesli model Evalautor stwierdzi ze odpowiedz agenta jest błedna, do reruna dodajmy do instruction feedback z evalautora 


agent_rerun = Agent(

    name = 'rerun',
    instructions= SCORING_INSTRUCTIONS + "\n\nDODATKOWO: dostajesz FEEDBACK od weryfikatora do swojej poprzedniej oceny. Uwzględnij go i oceń ponownie, bardziej precyzyjnie.",
    tools= [scrape_website,save_partner],

    model = 'gpt-5.4-mini'

)

In [214]:
 # piszemy funkcje która odpali evalautora  FUNKCJA ODPALAJCAA GENTA
# evalautor jako argument przyjmuje url danej domeny i reply czyli odpowiedz 1 agenta_score a wlasciwie jego return czyli str z odpowiedzią, url tez rpzeakzujemy z agemt_score
# evalautor odpala się w agent score
#rovimy funkcje asychroniczne zeby nie wydluzac dzialania kodu, ten bedzie działał dalej i nie musi czekac na odpowiedz modelu
# zeby zrobic tka funkcje musimy ja zrobic słowem async zeby potem uzuac await i zeby funkcja serio była asychroniczna dzialal w tle i nieblokowkwa reszty kodu do czasuu wykonania 
async def evaluator(url,result):
    # piszemy prompt który przeakzemy do agenta sklejamy w prompt informacje oraz url i result które dostanie evalautor przy odpaleniu w funkcji agent_score # nie duplikujemy evalutor prompt ten juz jest podany w EVALUATOR_PROMPT w agencie agennt
    # evalautor w istruction
    system_prompt= f"""jesteś agentem evalautor, weryfikujesz na podstawie wytycznych  odpowiedz moedlu w {result} , oceniasz {url}"""

    # odpalamya genta asycnhronicznie
    result = await Runner.run(agent_evaluator, system_prompt) 

    # zwracamy result do nzzwy funkcji dzieki czemu trafi do cklass evaluation

    return result.final_output

In [215]:
# FUNKCJA ODPALAJCAA GENTA
# piszemya sychnor niczna funkcje rerum która odpala sie jesli evalautor uznal ze odpwoeidz agent_score jest bledan wówczas przekazujemy fedbak z evalutora konkretnie z evalution class który przerabia odpoiwedz evalautora
# i jeszcze raz wzbogaceni o info z evlautora porpawiamy odpowedz modelu dzueki ulepszonmum  promptom
# funkcja odpala sie dopiero w agent_score pod wyzszymi warunkami przekazujemy jej url i feedback z evalutora
async def rerun(url,feedback):
    # piszemy system prompt
    system_prompt = f"""masz jeszcze raz ocenic url zgodnie z promptem, tu masz url strony {url} i  feedback z evalutora{feedback}"""

    result = await Runner.run(agent_rerun, system_prompt)

    return result.final_output

In [216]:

# teraz piszemy funkcje agent_score która przyjmuje jako arguemnt url, odpala asychronicznie agent_score i przesyla odpowiedz modelu do evaluator model odpalajac go i przeakzujac 
# swoja odpowiedz w result evlautor uruchmaia sie przekazuje odpiwedz do evalution class a potem jego zmiennna is_acceotable która moze byc true jesli model agent_score dał dobrą odpowiedz
# i false jesli evaluator agent zle ocenił odpowiedz wówczas class evlautron da is_acceotable false # i model uruchomi rerun

async def run(url):
    # tu juz nie tworzymy nowego system prompt rtylko przeakzujemy url przy wywołaniu

    # uruchomienie agent_score
    result = await Runner.run(agent,url)
    # odpowiedz z result z final_outpost metoda która zweira stringa z odpowiedz amodelu przekazujemy do zmienenj która posluzy nam do przekazania odpowedzi agent_score do evlautora
    reply = result.final_output


# teraz nie ruchamiamy agenta a funkcje!!!! evalutor, po to by zwróciła do zminnej cały obiekt evalution, a wlasiwie klase z 2 zmiennymi, my potrzbujemy is_acceptavble
# dzieki temy jeslu jest true to nic sie nie zadzieje a jesli jest false to odpali reruna 
# nie uruchmaiamy agenta agent jest juz w srodku funkcji !!     result = await Runner.run(agent_evaluator, system_prompt) 
 # jesli odpalenie agenta jest w fukcji TO ODPALANIE FUNKCJI ODPALA AGENTA  funkcje odpalamy z arugmentami!! jako 1 przekazujemy url jako 2 reply czyli odpowiedz 1 agenta która musi zostac sprawdzona
    evaluation_obj = await evaluator(url,reply)
    # robimy na pdostawie evalution obj i jego metody is_acceptale  która jest boolean
    if evaluation_obj.is_acceptable:
        # zwracamy reply 
        return reply
    else:# jesli nie urchmaiamy rerun z url i feecbak z evaluton_obj
       reply =  await rerun(url,evaluation_obj.feedback)
    return reply

In [217]:
result1 = await run('https://www.tebim.pro/')

print(result1)

- **Score:** 9/10  
- **Kontaktować:** TAK  
- **Kategoria:** Web dev / software house / twórcy sklepów (e-commerce)  
- **Model:** oba (referral + white-label)  
- **SEO/SEM/GEO w ofercie:** tak, ale jako element wspierający wdrożenia; **nie konkurent**, raczej **sub-partner**  
- **Profil klientów:** sklepy e-commerce i B2B, raczej firmy z budżetem, potrzebujące wdrożeń, integracji i rozwoju sprzedaży  
- **Kontakt:** `kontakt@tebim.pl`, `62 727 23 90`

**Uzasadnienie:**  
Tebim to mocny partner e-commerce: agencja PrestaShop 3 poziomu, 14 lat doświadczenia, 104 wdrożone sklepy i 132 klientów.  
Ich oferta obejmuje wdrożenia, B2B, integracje ERP/PIM/CRM/MA, UX/UI i utrzymanie, więc naturalnie pasują do referralów na SEO/SEM dla sklepów.  
To nie jest bezpośredni konkurent Last Agency — bardziej partner technologiczny do wspólnych projektów i obsługi klientów po wdrożeniu.
